In [ ]:
# Cell 1: Environment Setup

include("scripts/buildaux_helpers.jl")
include("scripts/buildaux_dictionaries.jl")
using .BuildAuxHelpers
using .BuildAuxDictionaries
using OMJulia

# --- Configuration ---

# 1. Directory containing the single-file model
MODEL_DIR = abspath("models")

# 2. Select the model to build
MODEL = "BESSloadAB"

# 3. Path to the selected model file
MODEL_FILE_PATH = joinpath(MODEL_DIR, MODEL * ".mo")

# 4. Path to the Dynawo package.mo
DYNAWO_PKG_PATH   = "/home/dyvulgawocfc/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"

# 5. Path to the Modelica package.mo
MODELICA_PKG_PATH = "/home/dyvulgawocfc/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"


In [ ]:
# Cell 2: OpenModelica Setup + Single Model Validation

# 1. Start OMC and load libraries
omc = OMJulia.OMCSession()
om_send(omc, "loadFile(\"$MODELICA_PKG_PATH\")")
om_send(omc, "loadModel(Complex)")
om_send(omc, "loadModel(ModelicaServices)")
om_send(omc, "loadFile(\"$DYNAWO_PKG_PATH\")")

# 2. Load the selected model and validate it
om_send(omc, "loadFile(\"$MODEL_FILE_PATH\")")
om_send(omc, "clearMessages()")
chk = om_send(omc, "checkModel($MODEL)", parsed=false)
println(chk)


In [ ]:
# Cell 3: Auxiliary Model Setup

AUX_MODEL = MODEL * "_auxiliary"
AUX_FILE = joinpath(MODEL_DIR, AUX_MODEL * ".mo")


In [ ]:
# Cell 4: INIT / Optional Slack Configuration for the Single Model

# Leave empty to disable slack-specific handling.
INIT_MODEL_BY_COMPONENT = Dict{String, String}(
  # "generatorSynchronous" => "GeneratorSynchronousInt_INIT",
)

SLACK_COMPONENT = ""


In [ ]:
# Cell 5: Single-Model Auxiliary Build Pipeline

# Create/refresh the auxiliary model in OpenModelica
om_send(omc, "deleteClass($AUX_MODEL)")
om_send(omc, "clearMessages()")
om_send(omc, "copyClass($MODEL, \"$AUX_MODEL\")")

# Build component dictionary from the source model
components = get_all_components(omc, MODEL)

# Apply dictionary-driven replacements
apply_replacements!(omc, MODEL, AUX_MODEL, REPLACEMENTS, components, SLACK_COMPONENT)

# Delete connections to cleanup targets
delete_connections!(omc, AUX_MODEL, components)

# Delete cleanup-target components
delete_components!(omc, AUX_MODEL, components)

# Add INIT models for the source model
add_init_models!(omc, MODEL, AUX_MODEL, INIT_MODELS, INIT_MODEL_BY_COMPONENT, components, SLACK_COMPONENT)

# Add load-flow modifiers
apply_LF_modifiers!(omc, MODEL, AUX_MODEL, INIT_MODELS, components)

# Add initial equations for the source model
add_init_equations!(omc, AUX_MODEL, components, INIT_MODELS, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)

# Save the auxiliary model
om_send(omc, "saveModel(\"$AUX_FILE\", $AUX_MODEL)")
patch_aux_equations!(AUX_FILE)

# Re-load the patched auxiliary model and validate the build
om_send(omc, "deleteClass($AUX_MODEL)")
om_send(omc, "loadFile(\"$AUX_FILE\")")
om_send(omc, "clearMessages()")
chk = om_send(omc, "checkModel($AUX_MODEL)", parsed=false)
println(chk)


### Optional diagnostics for the single-file build
Run the next cell only if the main build/check cell fails or you need detailed OpenModelica messages.


In [ ]:
# Cell 7: OMC diagnostics for failed checks

# Run this cell after the build/check cell to isolate OpenModelica failures.

function _print_omc_errors(label::String)
    raw = String(sendExpression(omc, "getErrorString()", parsed=false))
    txt = strip(replace(raw, "\"" => ""))
    println("\n[$label] getErrorString()")
    if isempty(txt)
        println("<no messages>")
    else
        println(raw)
    end
end

function _check_and_report(model_name::String)
    sendExpression(omc, "clearMessages()")
    println("\n=== checkModel($model_name) ===")
    chk = sendExpression(omc, "checkModel($model_name)", parsed=false)
    println(chk)
    _print_omc_errors(model_name)
    return chk
end

println("=== OMC diagnostics start ===")
_print_omc_errors("after previous cell")

# 1) Auxiliary model
_check_and_report(AUX_MODEL)

# 2) Optional compile-time expansion (often gives clearer errors)
sendExpression(omc, "clearMessages()")
println("\n=== instantiateModel($AUX_MODEL) ===")
inst = sendExpression(omc, "instantiateModel($AUX_MODEL)", parsed=false)
inst_s = String(inst)
if startswith(strip(inst_s), "Error")
    println(inst_s)
end
_print_omc_errors("instantiateModel")

println("=== OMC diagnostics end ===")
